**This code is to use shape descriptors for classification using multi-layer perceptrons (MLPs):**

In [10]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, FastICA, NMF, TruncatedSVD
from sklearn.random_projection import GaussianRandomProjection, SparseRandomProjection
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [7]:
# ---------------- CONFIG ----------------
MODEL = "padc_bee_dim_search"
DEVICE_ID = 0
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
EXPLAINED_VARIANCE_THRESHOLD = 0.99

DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
LOG_DIR = f"./{MODEL}_logs/"
train_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/Beemachine/bee_gt_shape_features_concise/bee_gt_features_train.csv"
val_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/Beemachine/bee_gt_shape_features_concise/bee_gt_features_val.csv"
test_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/Beemachine/bee_gt_shape_features_concise/bee_gt_features_test.csv"

device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [6]:
# ---------------- FAST IMPUTATION ----------------
def fill_by_class_mean(df, class_col="label"):
    df = df.copy()

    # Replace zeros with NaN
    df.replace(0, np.nan, inplace=True)

    # Keep only columns that are not all NaN
    df = df.dropna(axis=1, how="all")

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Compute class means once
    class_means = df.groupby(class_col)[numeric_cols].transform("mean")

    # Fill NaNs with class means
    df[numeric_cols] = df[numeric_cols].fillna(class_means)

    # Global fallback
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    return df

# ---------------- DATASET ----------------
class ShapeFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# The MLP classifier Network
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        mid_dim = input_dim
        if (input_dim // 2) > num_classes:
            mid_dim = input_dim // 2
        self.net = nn.Sequential(
            nn.Linear(input_dim, mid_dim),
            nn.LayerNorm(mid_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

def reduce_dimensions(X_train, X_val, X_test, method="PCA", n_components=100, random_state=SEED):
    """
    Apply dimensionality reduction on the dataset.
    Supported methods: "PCA", "ICA", "NMF", "TruncatedSVD", "GRP", "SRP"
    """
    if method == "PCA":
        reducer = PCA(n_components=n_components, random_state=random_state)
    elif method == "ICA":
        reducer = FastICA(n_components=n_components, random_state=random_state, max_iter=1000)
    elif method == "NMF":
        # NMF requires non-negative input
        X_train_nonneg = np.clip(X_train, 0, None)
        X_val_nonneg = np.clip(X_val, 0, None)
        X_test_nonneg = np.clip(X_test, 0, None)
        reducer = NMF(n_components=n_components, init='nndsvda', random_state=random_state, max_iter=1000)
        X_train = reducer.fit_transform(X_train_nonneg)
        X_val = reducer.transform(X_val_nonneg)
        X_test = reducer.transform(X_test_nonneg)
        return X_train, X_val, X_test, reducer
    elif method == "TruncatedSVD":
        reducer = TruncatedSVD(n_components=n_components, random_state=random_state)
    elif method == "GRP":
        reducer = GaussianRandomProjection(n_components=n_components, random_state=random_state)
    elif method == "SRP":
        reducer = SparseRandomProjection(n_components=n_components, random_state=random_state)
    else:
        raise ValueError(f"Unknown dimensionality reduction method: {method}")
    
    X_train_reduced = reducer.fit_transform(X_train)
    X_val_reduced   = reducer.transform(X_val)
    X_test_reduced  = reducer.transform(X_test)
    
    return X_train_reduced, X_val_reduced, X_test_reduced, reducer

def select_n_components_via_pca(X_train, explained_var_threshold=0.95, random_state=SEED, plot=False):
    """Use PCA to select optimal n_components based on explained variance."""
    pca_full = PCA(n_components=X_train.shape[1], random_state=random_state)
    pca_full.fit(X_train)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    n_components_opt = np.argmax(cumulative_variance >= explained_var_threshold) + 1
    print(f"Optimal n_components (PCA, {explained_var_threshold*100}% variance): {n_components_opt}")
    
    # Optional plot
    if plot:
        plt.figure(figsize=(8,5))
        plt.plot(cumulative_variance, marker='o')
        plt.axhline(y=explained_var_threshold, color='r', linestyle='--')
        plt.xlabel('Number of components')
        plt.ylabel('Cumulative explained variance')
        plt.title('PCA Explained Variance')
        plt.grid(True)
        plt.show()    
    return n_components_opt

# Training functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [11]:
train_df = pd.read_csv(train_path) 
val_df = pd.read_csv(val_path) 
test_df = pd.read_csv(test_path)

# Class names of every image
fulldata_path = r"/home/c/choton/beemachine/datasets/Bee_Full_Body_Segments/Images"

# Get list of classes
classes = os.listdir(fulldata_path)
species_dict = {}

for c in classes:
    images_path = os.path.join(fulldata_path, c)
    images = os.listdir(images_path)
    for image in images:
        image_name = image[:-4]
        species_dict[image_name] = c
image_names = list(species_dict.keys())
len(image_names)

8029

In [12]:
print(f"Shape of dataset, train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}")

Shape of dataset, train: (34722, 938), val: (1158, 938), test: (771, 938)


In [13]:
count = 0
species_y = {}
for image_n in tqdm(image_names):
    for image_y in train_df['image']:
        # print(image_n)
        # print(image_y)
        # input()
        image_nx = image_n.replace("._", "-_")
        image_nx = image_nx.replace(".jpg", "")
        if image_nx in image_y:
            count += 1
            species_y[image_y] = species_dict[image_n]
print(count)
train_df['species'] = train_df['image'].map(species_y)
train_df

100%|██████████| 8029/8029 [02:34<00:00, 52.07it/s]


34866


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4330.0,608.61730,3.586139,0.248822,0.568018,0.960334,1.295621,0.146896,0.721149,...,6.401090,60.094750,27.640350,41.000250,0.143003,23.202300,0.120566,0.843097,0.036337,Bombus_impatiens
1,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4325.0,606.82440,3.585948,0.248535,0.568854,0.960330,1.296722,0.147594,0.721134,...,6.392323,60.289352,24.562760,43.832634,0.142942,14.504794,0.117950,0.825161,0.056889,Bombus_impatiens
2,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4327.0,606.82440,3.584670,0.248650,0.569117,0.960301,-1.296543,0.147663,0.721034,...,6.391944,47.546585,27.722122,44.148280,0.143041,23.056402,0.120566,0.842877,0.036557,Bombus_impatiens
3,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4328.0,608.61730,3.587408,0.248707,0.567755,0.960363,-1.295801,0.146828,0.721247,...,6.400627,47.292450,24.678831,41.000793,0.142928,14.551177,0.117961,0.825320,0.056718,Bombus_impatiens
4,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4332.0,608.61730,3.587018,0.248937,0.568206,0.960354,0.274950,0.146964,0.721217,...,6.398315,56.920498,27.667301,43.878483,0.143164,23.063263,0.120658,0.842799,0.036543,Bombus_impatiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34717,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4248.0,488.43964,1.840455,0.564819,0.622965,0.839510,1.436783,0.223755,0.456656,...,9.348055,35.663630,25.897465,37.843678,0.106854,3.173799,0.075147,0.703267,0.221585,Bombus_zonatus
34718,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4248.0,488.43964,1.840455,0.564819,0.622965,0.839510,-1.436783,0.223755,0.456656,...,9.316640,53.308210,26.743814,37.511600,0.106854,3.173799,0.075147,0.703267,0.221585,Bombus_zonatus
34719,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4242.0,489.02542,1.842837,0.564021,0.622268,0.839964,-1.435658,0.222904,0.457359,...,9.353924,52.911057,25.226418,37.833157,0.106682,3.176212,0.075048,0.703471,0.221481,Bombus_zonatus
34720,Bombus_zonatus_GBIF_iNat_3058832608_2-jpg_jpg....,4245.0,488.43964,1.842244,0.564420,0.622708,0.839851,0.134523,0.223597,0.457184,...,9.333622,37.994225,25.915386,35.862720,0.106760,3.175371,0.075094,0.703391,0.221515,Bombus_zonatus


In [14]:
count = 0
species_y = {}
for image_n in tqdm(image_names):
    for image_y in val_df['image']:
        # print(image_n)
        # print(image_y)
        # input()
        image_nx = image_n.replace("._", "-_")
        image_nx = image_nx.replace(".jpg", "")
        if image_nx in image_y:
            count += 1
            species_y[image_y] = species_dict[image_n]
print(count)
val_df['species'] = val_df['image'].map(species_y)
val_df

100%|██████████| 8029/8029 [00:05<00:00, 1442.36it/s]

1161


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,5410.0,434.57568,1.484547,0.497243,0.739172,0.739091,-1.247538,0.359978,0.326394,...,6.574201,69.938730,28.917168,43.730812,0.180586,16.606430,0.145540,0.805929,0.048531,Bombus_flavifrons
1,0090L09000N0Q0SQQ080FQFKCQHQ1R3K1RIQH0N0L0SQOR...,2058.0,449.34525,1.546949,0.172564,0.321864,0.762970,0.208325,0.128084,0.353566,...,4.861553,66.378494,25.980085,47.686066,0.129605,1.430412,0.070873,0.546835,0.382292,Bombus_terricola
2,0H6RLHERTZIZ2LRZWLZZ6LYLELYL2LHZFZ0R2L0RRHSRRH...,3315.0,355.98990,1.162912,0.458697,0.689333,0.510445,-1.497345,0.328714,0.140089,...,7.166333,40.585583,25.544678,38.450874,0.109363,3.333187,0.077596,0.709534,0.212870,Bombus_griseocollis
3,0HGRDZIROZ7RFZXRULHZ9L0R3ZYLNLKROZRZ9LYLNLZZTZ...,608.0,173.63351,8.116632,0.405063,0.683914,0.992381,1.451794,0.253423,0.876796,...,6.655105,32.683594,25.874298,41.878220,0.018505,72.210990,0.017925,0.968661,0.013414,Bombus_vosnesenskii
4,0K9KLKWKIKT0UQLSPQA0UQB0GQB04Q6K2Q2K6QUKPQC0KK...,8.0,8.00000,2.236068,1.000000,1.000000,0.894427,1.570796,1.570796,0.552786,...,5.211917,60.000885,23.613640,57.017180,0.000322,10.280810,0.000293,0.911087,0.088620,Bombus_melanopygus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153,Bombus_sylvestris_iNat_21848423_1-jpg_jpg.rf.0...,3480.0,467.93713,1.185989,0.272492,0.429100,0.537634,-0.206154,0.199717,0.156822,...,7.189582,62.782967,24.507560,58.299995,0.142676,1.522059,0.079278,0.555654,0.365067,Bombus_sylvestris
1154,Bombus_sylvestris_iNat_27771888_01-jpg_jpg.rf....,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.752216,89.865560,26.728430,69.970680,0.000000,0.890869,0.000000,0.471143,0.528857,Bombus_sylvestris
1155,Bombus_sylvestris_iNat_36837121_01-jpg_jpg.rf....,4070.0,504.61120,1.798342,0.339167,0.596337,0.831137,0.148295,0.200859,0.443932,...,7.327914,60.891003,27.185130,51.417065,2.110996,0.075445,0.128989,0.061104,0.809907,Bombus_sylvestris
1156,Bombus_sylvestris_iNat_47607375_1-jpg_jpg.rf.3...,5225.0,453.34525,1.978194,0.424797,0.698810,0.862820,-1.515664,0.319476,0.494488,...,7.893108,30.526089,28.071922,33.448742,1.948173,0.108028,0.159620,0.081933,0.758447,Bombus_sylvestris


In [15]:
count = 0
species_y = {}
for image_n in tqdm(image_names):
    for image_y in test_df['image']:
        # print(image_n)
        # print(image_y)
        # input()
        image_nx = image_n.replace("._", "-_")
        image_nx = image_nx.replace(".jpg", "")
        if image_nx in image_y:
            count += 1
            species_y[image_y] = species_dict[image_n]
print(count)
test_df['species'] = test_df['image'].map(species_y)
test_df

100%|██████████| 8029/8029 [00:02<00:00, 3087.32it/s]


774


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,0090L0W0TQIQURLQ9RSQ3RKQTRQQOR7Q3R60CQG0OQ80FQ...,4279.0,506.32693,1.777138,0.443144,0.666926,0.826659,-1.310267,0.209744,0.437298,...,6.121056,63.953552,29.595713,47.932377,0.198838,1.826825e+00,0.113867,0.572660,0.313473,Bombus_crotchii
1,0090Q0W0H0X0YQYKWR40S0SQURI0JQ70ARZQAR70CQHQUR...,2892.0,221.48022,1.697738,0.708824,0.953511,0.808119,-0.267619,0.740864,0.410981,...,7.187157,72.190110,29.266737,54.327305,0.088424,8.546120e+00,0.073354,0.829575,0.097070,Bombus_vagans
2,0K6KRKPKIKOK4KVKVQO08QPKKKD05QD0EQ9KSKA04QC04Q...,9156.0,400.45080,1.215336,0.726667,0.960151,0.568306,0.103841,0.717492,0.177182,...,5.984114,69.195910,26.606197,63.917778,0.377038,2.611183e+02,0.273044,0.724182,0.002773,Bombus_flavidus
3,0K9KLKWKIKT0UQB0PQLSWQF0NQY05KAKVQY05QC05Q6KQK...,2169.0,453.03763,1.134231,0.156178,0.303145,0.471896,1.255855,0.132801,0.118346,...,5.446256,79.839950,25.544370,59.388435,0.096606,1.111485e+01,0.081416,0.842761,0.075823,Bombus_perplexus
4,0L6ZLLEZILTH2HAHIHDHUHYHNHZREHJH7LNZMLFHWHVZQL...,7078.0,669.78890,1.372911,0.538006,0.701695,0.685174,-0.135562,0.198264,0.271621,...,6.088460,67.306000,27.478662,61.603825,0.247439,2.860500e+10,0.198358,0.801642,0.000000,Bombus_flavidus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,Bombus_trifasciatus_iNat_95727364_01-jpg_jpg.r...,7539.0,592.03760,2.363889,0.505159,0.795589,0.906115,-1.441886,0.270287,0.576968,...,7.466583,53.445713,25.307562,48.816837,20.542234,1.919255e-02,0.278933,0.013579,0.707489,Bombus_trifasciatus
767,Bombus_zonatus_GBIF_iNat_2236827425_2-jpg_jpg....,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,NaN,27.075386,100.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,Bombus_zonatus
768,Bombus_zonatus_GBIF_iNat_2236827425_3-jpg_jpg....,4149.0,622.22240,1.287401,0.235124,0.330361,0.629799,1.463304,0.134667,0.223242,...,4.817892,63.231586,26.097486,42.069763,0.334867,1.071614e+00,0.147646,0.440910,0.411444,Bombus_zonatus
769,Bombus_zonatus_GBIF_iNat_3058832608_1-jpg_jpg....,4461.0,550.49960,1.795978,0.319579,0.517457,0.830647,0.541462,0.184981,0.443200,...,7.733049,58.705130,27.434587,42.797490,1.946335,1.165463e-01,0.168856,0.086756,0.744389,Bombus_zonatus


In [17]:
train_df = fill_by_class_mean(train_df, class_col="species")
val_df   = fill_by_class_mean(val_df, class_col="species")
test_df  = fill_by_class_mean(test_df, class_col="species")

In [18]:
X_train = train_df.drop(columns=["image", "species"])
X_val = val_df.drop(columns=["image", "species"])
X_test = test_df.drop(columns=["image", "species"])
y_train = train_df["species"]
y_val = val_df["species"]
y_test = test_df["species"]

# --- Sanity check ---
assert not X_train.isna().any().any(), "NaNs remain in X_train!"
assert not X_val.isna().any().any(), "NaNs remain in X_val!"
assert not X_test.isna().any().any(), "NaNs remain in X_val!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

# ----------------- Standardize (important!) -----------------
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train.values)
X_val_std  = scaler.transform(X_val.values)
X_test_std = scaler.transform(X_test.values)

# ----------------- Class mapping (use same mapping for train & val) -----------------
classes = sorted(y_train.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

y_train_idx = y_train.map(class_to_idx).values
y_val_idx   = y_val.map(class_to_idx).values
y_test_idx = y_test.map(class_to_idx).values

✅ Class-wise NaN imputation complete — no missing values remain.


In [19]:
# Fit PCA with all components
pca_full = PCA(n_components=X_train_std.shape[1], random_state=SEED)
pca_full.fit(X_train_std)

# Cumulative explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components to explain desired variance (e.g., 95%)
explained_threshold = EXPLAINED_VARIANCE_THRESHOLD
n_components_opt = np.argmax(cumulative_variance >= explained_threshold) + 1

print(f"Optimal n_components for {explained_threshold*100}% variance: {n_components_opt}")

Optimal n_components for 99.0% variance: 637


In [22]:
methods = ["PCA", "ICA", "TruncatedSVD", "GRP", "SRP", "NMF"]
best_acc = -1
best_method = None
best_data = None

device = torch.device(f"cuda:{DEVICE_ID}")

summary_rows = []

for method in methods:
    print(f"\n================ {method} ================\n")

    Xtr, Xva, Xte, reducer = reduce_dimensions(
        X_train_std, X_val_std, X_test_std,
        method=method,
        n_components=n_components_opt
    )

    train_dataset = ShapeFeatureDataset(Xtr, y_train_idx)
    val_dataset   = ShapeFeatureDataset(Xva, y_val_idx)
    test_dataset  = ShapeFeatureDataset(Xte, y_test_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MLPClassifier(Xtr.shape[1], len(classes), dropout=DROPOUT).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        device, EPOCHS,
        LOG_DIR + f"_{method}"
    )

    # ---- load best checkpoint ----
    best_path = os.path.join(LOG_DIR + f"_{method}", "checkpoints", "best_model.pth")
    model.load_state_dict(torch.load(best_path, map_location=device))

    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
    test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)

    print(f"{method} Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")

    summary_rows.append({
        "method": method,
        "original_dim": X_train_std.shape[1],
        "reduced_dim": Xtr.shape[1],
        "val_acc": val_acc,
        "test_acc": test_acc
    })

    if test_acc > best_acc:
        best_acc = test_acc
        best_method = method
        best_data = (Xtr, Xva, Xte)


================ PCA ================

Starting the first epoch...
 Epoch 1/100 | Train Loss: 4.3601 | Train Acc: 0.1372 | Val Loss: 3.9432 | Val Acc: 0.1986
 Epoch 2/100 | Train Loss: 3.2577 | Train Acc: 0.3295 | Val Loss: 3.6860 | Val Acc: 0.2573
 Epoch 3/100 | Train Loss: 2.6106 | Train Acc: 0.5016 | Val Loss: 3.5761 | Val Acc: 0.2703
 Epoch 4/100 | Train Loss: 2.1372 | Train Acc: 0.6512 | Val Loss: 3.5524 | Val Acc: 0.2841
 Epoch 5/100 | Train Loss: 1.8145 | Train Acc: 0.7602 | Val Loss: 3.5768 | Val Acc: 0.2703
 Epoch 6/100 | Train Loss: 1.6033 | Train Acc: 0.8322 | Val Loss: 3.6123 | Val Acc: 0.2755
 Epoch 7/100 | Train Loss: 1.4638 | Train Acc: 0.8822 | Val Loss: 3.6684 | Val Acc: 0.2686
 Epoch 8/100 | Train Loss: 1.3727 | Train Acc: 0.9147 | Val Loss: 3.6847 | Val Acc: 0.2703
 Epoch 9/100 | Train Loss: 1.3316 | Train Acc: 0.9285 | Val Loss: 3.7065 | Val Acc: 0.2677
 Epoch 10/100 | Train Loss: 1.3025 | Train Acc: 0.9370 | Val Loss: 3.7255 | Val Acc: 0.2694
 Epoch 11/100 | Train

/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/sklearn/decomposition/_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Starting the first epoch...
 Epoch 1/100 | Train Loss: 4.5778 | Train Acc: 0.1023 | Val Loss: 4.3045 | Val Acc: 0.1494
 Epoch 2/100 | Train Loss: 3.3825 | Train Acc: 0.3238 | Val Loss: 3.9878 | Val Acc: 0.2081
 Epoch 3/100 | Train Loss: 2.5801 | Train Acc: 0.5447 | Val Loss: 3.8066 | Val Acc: 0.2478
 Epoch 4/100 | Train Loss: 2.0352 | Train Acc: 0.6994 | Val Loss: 3.7741 | Val Acc: 0.2487
 Epoch 5/100 | Train Loss: 1.7095 | Train Acc: 0.8013 | Val Loss: 3.7801 | Val Acc: 0.2444
 Epoch 6/100 | Train Loss: 1.5206 | Train Acc: 0.8625 | Val Loss: 3.8139 | Val Acc: 0.2453
 Epoch 7/100 | Train Loss: 1.4026 | Train Acc: 0.9023 | Val Loss: 3.8489 | Val Acc: 0.2547
 Epoch 8/100 | Train Loss: 1.3292 | Train Acc: 0.9259 | Val Loss: 3.8656 | Val Acc: 0.2487
 Epoch 9/100 | Train Loss: 1.2939 | Train Acc: 0.9379 | Val Loss: 3.8822 | Val Acc: 0.2435
 Epoch 10/100 | Train Loss: 1.2698 | Train Acc: 0.9455 | Val Loss: 3.8983 | Val Acc: 0.2470
 Epoch 11/100 | Train Loss: 1.2432 | Train Acc: 0.9569 | Val 

In [23]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("bee_dim_reduction_summary.csv", index=False)

print("\nSaved summary to dim_reduction_summary.csv")
print(summary_df)


Saved summary to dim_reduction_summary.csv
         method  original_dim  reduced_dim   val_acc  test_acc
0           PCA           937          637  0.284111  0.236057
1           ICA           937          637  0.248705  0.198444
2  TruncatedSVD           937          637  0.267703  0.226978
3           GRP           937          637  0.241796  0.207523
4           SRP           937          637  0.246978  0.223087
5           NMF           937          637  0.193437  0.163424


In [24]:
print(f"\n🏆 Best method: {best_method} (Test Acc={best_acc:.4f})")

Xtr_best, Xva_best, Xte_best = best_data

cols = [f"{best_method}_comp_{i}" for i in range(Xtr_best.shape[1])]

train_out = pd.DataFrame(Xtr_best, columns=cols)
train_out["label"] = y_train

val_out = pd.DataFrame(Xva_best, columns=cols)
val_out["label"] = y_val

test_out = pd.DataFrame(Xte_best, columns=cols)
test_out["label"] = y_test

train_out.to_csv(f"bee_reduced_train_{best_method}.csv", index=False)
val_out.to_csv(f"bee_reduced_val_{best_method}.csv", index=False)
test_out.to_csv(f"bee_reduced_test_{best_method}.csv", index=False)

print("✅ Saved reduced datasets for best method.")


🏆 Best method: PCA (Test Acc=0.2361)
✅ Saved reduced datasets for best method.
